# Cooking, gate gfail - Qwen3-1.7B, 4B, 8B
Plays `grids/cooking_fail.yaml`: the gate `gfail` (think when the game refused the last command), `lgfail` (the same, keeping the last reasoning) and the random gates `grand25` / `grand50`, prompt `CoT+nonempty`, cooking levels 1-3-5-7-9, seeds 1-10, for the three Qwen3 sizes one after the other (see the grid's comment and THESIS_LOG.md, 2026-11-11).
The 8B uses its fp8 KV cache here, as its cooking games already played (the bf16 cache is tight for cooking contexts).
This gate has only been tested with a scripted model: look at the first games (`limit = 4`) before leaving the whole grid running.

# Setup
Clones the repository (only the folders needed), installs TextWorld and vLLM and moves into `Code/Benchmark`. Locally it does nothing.
The first time on a new Colab machine the installation takes a few minutes: restart the kernel after it (and then run all the cells), because it can update torch.

In [1]:
import os
import sys
import shutil
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/UPROOT01/LLMs-Play-Textual-Games.git"
    repo_dir = "/content/LLMs-Play-Textual-Games"
    if not os.path.isfile(f"{repo_dir}/Code/Benchmark/run.py"):
        shutil.rmtree(repo_dir, ignore_errors=True) # leftover of a failed clone, or a clone without the benchmark
        # sparse clone: only the folders needed (the whole Games folder is >1 GB)
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", repo_url, repo_dir], check=True)
    else:
        # the latest code and the latest results pushed to git (they decide what is skipped, with Drive)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    subprocess.run(["git", "-C", repo_dir, "sparse-checkout", "add", "Code/Benchmark",
                    "Code/Games/cooking", "Code/Games/coin_collector", "Code/Games/treasure_hunter"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "textworld", "vllm==0.31.0", "pyyaml"], check=True)
    os.chdir(f"{repo_dir}/Code/Benchmark")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

Working directory: /content/LLMs-Play-Textual-Games/Code/Benchmark


In [2]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import torch

assert torch.cuda.is_available(), "No GPU available: connect to a Colab runtime with a GPU (e.g. L4)"
! nvidia-smi

Wed Nov 11 11:40:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             14W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# What to run
The grid and the models are fixed for this notebook. `limit`: at most this many games per model (`None`: all; e.g. `4` for a first look).
`n_parallel = "auto"`: as many games at once as fit in the GPU memory (see `engine.ContextBudget`).

The first cell below prints, for each results folder, how many games of the grid it already holds: the games found in Drive **or** in the git clone are skipped. Results played on another machine are seen only once they are committed and pushed.

In [4]:
from records import LEGACY_RESULTS
from run import played_by_folder, summary
from run import load_grid, run_grid

grid = load_grid("grids/cooking_fail.yaml")
models = None
limit = None
n_parallel = "auto"

# Where the games already played are looked for: a game found in ANY of these folders is not played again.
#   - Drive: the games played on Colab (new games are saved here)
#   - the git clone (Code/Benchmark/results): the games committed and pushed from anywhere (the remote machine, other
#     Colab runs copied into the repo). The setup cell pulls before this, so it is the latest pushed state.
if IN_COLAB:
    assert os.path.isdir("/content/drive/MyDrive"), "Google Drive is not mounted: run the cell above"
    results = "/content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results"
else:
    results = "results"
read = [LEGACY_RESULTS] if os.path.abspath(LEGACY_RESULTS) != os.path.abspath(results) else []
print("New games saved in:", results)
print("Also read:", read, "\n")
played_by_folder(grid, [results, *read], models)
print()
summary(grid, results, read, models)

New games saved in: /content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results
Also read: ['/content/LLMs-Play-Textual-Games/Code/Benchmark/results'] 

/content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results: 0 games of the grid, 0 only here
/content/LLMs-Play-Textual-Games/Code/Benchmark/results: 0 games of the grid, 0 only here
played in at least one folder: 0 of 600

model                                          mode  played  left  ~Mtokens left
Qwen/Qwen3-1.7B                               gfail       0    50               
Qwen/Qwen3-1.7B                              lgfail       0    50               
Qwen/Qwen3-1.7B                              grand25       0    50               
Qwen/Qwen3-1.7B                              grand50       0    50               
Qwen/Qwen3-4B                                 gfail       0    50               
Qwen/Qwen3-4B                                lgfail       0    50               
Qwen/Qwen3-4B                              

# Run
Every game is saved on Drive as soon as it ends. Stopping the cell is safe: the games being played are lost, the others are kept; run the notebook again to resume.
If loading the second model fails because the GPU memory is full, restart the runtime and run everything again: the first model's games are skipped.

In [5]:
run_grid(grid, results, read, models=models, limit=limit, n_parallel=n_parallel)
summary(grid, results, read, models)

Qwen/Qwen3-1.7B: 200 games to play


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (7 outputs left out)

Qwen/Qwen3-1.7B: KV cache of 143616 tokens


12 games running, 160 waiting, 28 ended; since the start 60.6 games/hour, 14243 tokens/min; last 10 min 48.1 games/hour, 15830 tokens/min; contexts 89% of the KV cache at their expected end

('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfail', 3, 1): loop, steps 35, score 3/11, 16 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'lgfail', 3, 1): loop, steps 30, score 0/11, 138 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'lgfail', 1, 1): loop, steps 29, score 0/5, 140 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfail', 1, 1): loop, steps 36, score 0/5, 145 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'grand25', 3, 1): steps 72, score 4/11, 135 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfail', 5, 1): steps 100, score 0/8, 24 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'lgfail', 5, 1): steps 100, score 0/8, 24 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfail', 7, 1): steps 100, score 0/11, 26 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'lgfail', 7, 1): steps 100, score 0/11, 27 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfail', 1, 2): steps 23, score 2/5, 153 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'grand25', 5, 1): s

KeyboardInterrupt: 

# Quick look
Per model and mode, from Drive and git together: games, average normalized score, thinking turns and thinking tokens per game. The baselines of the same levels and seeds (0, e1, er4, lr4, r4) are in the second table.

In [ ]:
from records import load_table

table = load_table(results, *read)
table["level"] = table.level.astype(str)
same = (table.game.isin(grid["games"]) & table.model.isin(models or grid["models"]) & table.seed.isin(grid["seeds"])
        & table.level.isin([str(level) for level in grid["games"]["cooking"]]))
per_game = dict(games=("seed", "size"), score=("norm_score", "mean"),
                thinks=("tokens_think_turns", "mean"), kthink=("tokens_generated_thinking", lambda t: t.mean() / 1000))
display(table[same & table["mode"].isin(grid["modes"]) & table.prompt.isin(grid["prompts"])]
        .groupby(["model", "mode"]).agg(**per_game).round(2))
table[same & table["mode"].isin(["0", "e1", "er4", "lr4", "r4"]) & table.prompt.isin(["CoT", "CoT+nonempty"])] \
    .groupby(["model", "prompt", "mode"]).agg(**per_game).round(2)